In [ ]:


base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false"
]

In [ ]:
base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false",
    "stars:>50 topic:android fork:false archived:false",
]

In [ ]:
base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false",
    "stars:>50 topic:android fork:false archived:false",
    "stars:>50 android fork:false archived:false"
]


def filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
    if item.get('stargazers_count', 0) <= 50:
        return False
    if item.get('fork', False) != expected_fork:
        return False
    if item.get('archived', False) != expected_archived:
        return False

    language = (item.get('language') or '').lower()
    topics = item.get('topics', [])
    description = (item.get('description') or '').lower()
    name = (item.get('name') or '').lower()

    if expected_language:
        return language == expected_language
    elif expected_topic:
        return expected_topic in topics
    elif expected_keyword:
        return expected_keyword in description or expected_keyword in name or expected_keyword in topics
    else:
        return True

In [ ]:
import os
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# === LOGGING ===
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# === CONFIG ===
output_dir = r"C:\Android Mobile App\Step 1_URL_Search_Kotlin\Kotlin,Java,Dart,android  (Fork false, Archived false)"
output_csv_raw = os.path.join(output_dir, "github_android_search_results_raw.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered.csv")

os.makedirs(output_dir, exist_ok=True)

# === AUTH ===
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
HEADERS = {
    "Authorization": f"token {tokens[token_index]}",
    "Accept": "application/vnd.github+json"
}

# === DATE RANGE ===
start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")

# === WINDOW SETTINGS ===
initial_window_hours = 15 * 24  # 10 days
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === BASE QUERIES ===
base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false",
    "stars:>50 topic:android fork:false archived:false",
    "stars:>50 android fork:false archived:false"
]

# === Filter ===
def filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
    if item.get('stargazers_count', 0) <= 50:
        return False
    if item.get('fork', False) != expected_fork:
        return False
    if item.get('archived', False) != expected_archived:
        return False

    language = (item.get('language') or '').lower()
    topics = item.get('topics', [])
    description = (item.get('description') or '').lower()
    name = (item.get('name') or '').lower()

    if expected_language:
        return language == expected_language
    elif expected_topic:
        return expected_topic in topics
    elif expected_keyword:
        return expected_keyword in description or expected_keyword in name or expected_keyword in topics
    else:
        return True

# === Token Rotate ===
def rotate_token():
    global token_index, HEADERS
    token_index = (token_index + 1) % len(tokens)
    HEADERS = {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github+json"
    }
    logger.warning(f"🔑 Rotated → Token #{token_index + 1} / {len(tokens)}")

# === Rate Limit ===
def check_rate_limit():
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("⚠️  Could not check rate limit.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())
    logger.info(f"🔎 Remaining: {remaining} | Resets in {reset_in/60:.1f} min")
    if remaining < 5:
        if len(tokens) > 1:
            rotate_token()
            check_rate_limit()
        else:
            logger.warning(f"⏳ No extra tokens → sleeping {reset_in/60:.1f} min")
            sleep(reset_in + 5)

# === Count ===
def check_count(query):
    while True:
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 1})
        if r.status_code == 200:
            return r.json().get("total_count", 0)
        elif r.status_code == 403:
            rotate_token()
        else:
            logger.error(f"❌ Count error: {r.status_code} — {r.text}")
            return -1

# === Fetch ===
def fetch_items(query):
    all_items = []
    for page in range(1, 11):
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 100, "page": page})
        if r.status_code == 200:
            items = r.json().get("items", [])
            if not items:
                break
            all_items.extend(items)
            sleep(1)
        elif r.status_code == 403:
            rotate_token()
            return fetch_items(query)
        else:
            logger.error(f"❌ Fetch error: {r.status_code} — {r.text}")
            break
    return all_items

# === MAIN ===
all_results = []
for base_query_prefix in base_queries:
    current_start = start_date
    window_hours = initial_window_hours

    while current_start < end_date:
        current_end = min(current_start + timedelta(hours=window_hours), end_date)
        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
        base_query = f"{base_query_prefix} {date_range}"

        total_count = check_count(base_query)
        logger.info(f"⏳ {base_query} → {total_count} repos")

        if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
            window_hours = max(window_hours // 2, min_window_hours)
            continue
        elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
            window_hours = min(window_hours * 2, max_window_hours)

        items = fetch_items(base_query)
        expected_fork = 'fork:true' in base_query_prefix
        expected_archived = 'archived:true' in base_query_prefix
        expected_language = ''
        expected_topic = ''
        expected_keyword = ''

        if 'language:Kotlin' in base_query_prefix:
            expected_language = 'kotlin'
        elif 'language:Java' in base_query_prefix:
            expected_language = 'java'
        elif 'language:Dart' in base_query_prefix:
            expected_language = 'dart'
        elif 'topic:android' in base_query_prefix:
            expected_topic = 'android'
        elif 'android' in base_query_prefix:
            expected_keyword = 'android'

        filtered = [
            item for item in items
            if filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword)
        ]
        for item in filtered:
            item['search_qualifier'] = base_query
            item['base_qualifier'] = base_query_prefix
            item['repo_stars'] = item.get('stargazers_count', 0)
            item['match_type'] = expected_language.capitalize() or expected_topic.capitalize() or expected_keyword.capitalize()

        all_results.extend(filtered)
        pd.json_normalize(all_results).to_csv(output_csv_raw, index=False)

        logger.info(f"✅ Window done: {len(filtered)} items")

        current_start = current_end + timedelta(seconds=1)

# === SAVE ===
df = pd.json_normalize(all_results)
keep_fields = [
    'language', 'base_qualifier', 'search_qualifier', 'match_type', 'name', 'full_name', 'private', 'html_url', 'url',
    'clone_url', 'visibility', 'owner.login', 'size', 'stargazers_count',
    'watchers_count', 'forks', 'open_issues', 'default_branch',
    'open_issues_count', 'repo_stars', 'topics', 'description', 'fork', 'archived'
]
for field in keep_fields:
    if field not in df.columns:
        df[field] = None

df = df[keep_fields]
df.to_csv(output_csv_raw, index=False)
df.to_csv(output_csv_filtered, index=False)
logger.info(f"✅ All done! Raw & filtered saved to: {output_csv_raw} and {output_csv_filtered}")
